# Microvilli Enrichment Analysis — Spherical Coordinate Receptor Profiling

*For Cai Lab NanoMAP data analysis workflow. Created by: En Cai. This version was edited on 2026/07/31, tested with **Python 3.13.5.***

**Loads multi-channel z-stack TIFF images, uses a pre-computed signed 3D distance transform channel,
and computes receptor channel intensity profiles in polar (spherical) coordinates per z-slice.**

**Input:**
- Multi-channel z-stack TIFF images (32-bit) exported from Imaris data file. 
- The Imaris file should contain: 
  - one membrane channel 
  - one or more receptor channels 
  - one signed distance transformation channel from cell surface
  - (optional) one mask channel for regions excluded from analysis (exclue: voxel_value=1; keep voxel_value=0)
- File naming convention: `{sample_name}_C0{channel}_Z000{z_slice}.tif`
- **User Configuration** in cell 2. 

**Output:**
- Fluorescent intensity profiling of membrane and receptor channels in **Polar Coordinate**. 
- Enrichment in microvilli for each receptor normalized to membrane at each z-slice and average value for selected z-slices of the a cell. 
  - Result value > 1 --> Enriched in microvilli
  - Result value = 1 --> No enrichment in microvilli (same as the membrane)
  - Result value < 1 --> Excluded from microvilli  
- Cell 15 **Output Summary** lists output files. 

**Distance-transform channel convention (C3):**
- Negative values → inside the cell
- Positive values → outside the cell  
- Zero-crossing  → cell surface boundary
- Units          → micrometres (µm)

## 1. Dependencies

In [ ]:
# Uncomment to install if needed:
# !pip install numpy scipy scikit-image matplotlib tifffile pandas joblib

import os, re, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — faster and thread-safe for file saving
matplotlib.rcParams["figure.max_open_warning"] = 0   # suppress warning in parallel loops
import matplotlib.pyplot as plt

import tifffile
from scipy.ndimage import label
from scipy.optimize import curve_fit
from skimage.measure import regionprops
from joblib import Parallel, delayed

print('All dependencies loaded.')

---
## 2. User Configuration
**Edit only this cell before running the analysis.**

In [ ]:
# ============================================================
#  USER CONFIGURATION
# ============================================================

# Path to the folder containing all TIFF files
DATA_FOLDER = './'   # <-- Set the path to your data folder here 

# --- Voxel size (micrometres) ---
VOXEL_XY_UM = 0.025    # lateral (x, y)
VOXEL_Z_UM  = 0.070    # axial   (z)

# --- Channel markers ---
# Integer key → biological marker name.
# The signed 3D distance-transform channel must be included.
CHANNEL_MARKERS = {
    0: "Receptor A",   # receptor channel
    1: "Receptor B",   # receptor channel
    2: "Membrane",   # receptor channel
    3: "Cell_DT",      # pre-computed signed 3D distance transform (µm)
}

# Channel index that holds the signed 3D distance transform
DT_CHANNEL = 3


# Channel index of the cell mask (pixel value 0 = keep, 1 = mask out).
# Set to -1 if there is no mask channel.
# Masked θ-bins (those whose ray passes through the dilated mask within
# DT_RANGE_UM are excluded from polar profiling and all downstream steps.
MASK_CHANNEL = -1

# Dilation buffer (pixels) added around mask=1 regions before projecting
# into polar space. 0 = no dilation (use mask exactly as-is).
MASK_DILATION_PX = 1

# Gaussian smooth sigma (pixels) applied to the DT channel (C3) before
# cell boundary detection.
# Larger values smooth out noise in the boundary; 0 = no smoothing.
# Typical range: 0.5 – 3.0 pixels.
DT_SMOOTH_SIGMA_PX = 0


# Receptor and membrane channels
MEMBRANE_CH  = 2  # channel index of the membrane marker
RECEPTORA_CH = 1   # channel index of receptor A  (of interest)
RECEPTORB_CH = 0   # channel index of receptor B  (comparison)



# Receptor channel indices to profile.
# Set to None to use all channels except DT_CHANNEL, MEMBRANE_CH, and MASK_CHANNEL.
RECEPTOR_CHANNELS = [0, 1, 2]

# Channel index range to load [min, max] inclusive. None = all found.
CHANNEL_RANGE = None

# Z-slice index range to load [min, max] inclusive. None = all found.
Z_SLICE_RANGE = [60, 80]  # <-- Set the z-slice range to load here 

# --- Z-stack smoothing (rolling-window intensity projection) ---
# WINDOW_SIZE = 1  → use original z-stack without projection
# WINDOW_SIZE = 3  → each output slice = projection of (z-1, z, z+1)
# WINDOW_SIZE = 5  → each output slice = projection of (z-2..z+2)
#
# PROJECTION_TYPE controls how slices in the window are combined:
#   'mean' → average-intensity projection
#   'max'  → maximum-intensity projection
WINDOW_SIZE     = 3
PROJECTION_TYPE = 'mean'   # 'mean' or 'max'

# --- Polar profile parameters ---

# Desired spatial resolution at the cell boundary (micrometres).
# THETA_STEP_DEG is computed automatically per z-slice from this value
# and the boundary radius of that specific slice (Cell 4).
#   arc_length = r_boundary(z) * dθ  →  dθ(z) = feature_size / r_boundary(z)
# The finest dθ across all z-slices is used globally so no slice is
# under-sampled (driven by the smallest-radius slice).
FEATURE_SIZE_UM = 0.12       # µm — angular step will match this arc length

# DT range to sample (µm). Negative = inside cell, positive = outside.
# Should span from -(max interior DT) to +(max exterior DT).
DT_RANGE_UM = (-0.15, 1.25)

# Number of DT bins = n * (DT range in pixels).
# n=1 → one bin per pixel (bin width = VOXEL_XY_UM)
# n=2 → two bins per pixel (finer, bin width = VOXEL_XY_UM / 2)
# n=0.5 → one bin per 2 pixels (coarser, bin width = VOXEL_XY_UM * 2)
DT_BINS_PER_PIXEL = 0.5

# Ray marching step (pixels along x-y spatial axes)
RADIAL_STEP_PX = 1

# Colormaps per receptor channel (cycles if more channels than entries)
CHANNEL_CMAPS = ["viridis", "plasma", "cividis", "magma", "inferno", "YlOrRd"]

# --- Parallelism ---
# Number of parallel workers for slow loops (profile computation, peak fitting).
# -1 = use all available CPU cores.  Set to 1 to disable parallelism.
# Recommended: set to the number of physical CPU cores (e.g. 18 for a 36-thread CPU).
N_JOBS = 1

# --- Peak detection parameters (Cell 10) ---
# Minimum absolute intensity for a peak to be detected (on normalized [0,1] profile).
# Use channel index as key; applies uniformly across all z-slices for that channel.

PEAK_MIN_INTENSITY = {
    0: 0.2,   # <-- Set the minimum peak intensity for Receptor A here
    1: 0.2,   # <-- Set the minimum peak intensity for Receptor B here
    2: 0.2,   # <-- Set the minimum peak intensity for membrane here
}

# DT threshold (µm) for classifying a detected peak center:
# |DT_centre| < BOUNDARY_TOL_UM  → 'boundary'
# DT_centre  >=+BOUNDARY_TOL_UM  → 'outside'
# DT_centre  <=-BOUNDARY_TOL_UM  → 'inside'
BOUNDARY_TOL_UM = 0.03

# Peak detection parameters (Cell 10)
# Minimum separation between detected peaks along the DT axis (µm).
PEAK_MIN_SEP_UM  = 0.05

# Keep only peaks whose height >= PEAK_REL_HEIGHT × height of the tallest peak.
# E.g. 0.30 means peaks below 30% of the major peak are discarded.
PEAK_REL_HEIGHT  = 0.30


# ============================================================
#  END OF USER CONFIGURATION
# ============================================================

# Partially-derived globals
_special_chs = {DT_CHANNEL, MEMBRANE_CH}
if MASK_CHANNEL >= 0:
    _special_chs.add(MASK_CHANNEL)
if RECEPTOR_CHANNELS is None:
    RECEPTOR_CHANNELS = [ch for ch in CHANNEL_MARKERS if ch not in _special_chs]

dt_span_um    = DT_RANGE_UM[1] - DT_RANGE_UM[0]
dt_span_px    = dt_span_um / VOXEL_XY_UM
N_DT_BINS     = max(1, int(round(dt_span_px * DT_BINS_PER_PIXEL)))
BIN_EDGES_UM  = np.linspace(DT_RANGE_UM[0], DT_RANGE_UM[1], N_DT_BINS + 1)
DT_BINS_UM    = 0.5 * (BIN_EDGES_UM[:-1] + BIN_EDGES_UM[1:])
bin_width_um  = dt_span_um / N_DT_BINS

OUTPUT_FOLDER   = os.path.join(DATA_FOLDER, "results")
FOLDER_SUMMARY  = os.path.join(OUTPUT_FOLDER, "summary")       # CSVs + z-stack-wide PNGs
FOLDER_POLAR    = os.path.join(OUTPUT_FOLDER, "polar_profile")  # per-slice polar PNGs
FOLDER_DIFF     = os.path.join(OUTPUT_FOLDER, "diff_profile")   # per-slice diff PNGs
FOLDER_PEAKS    = os.path.join(OUTPUT_FOLDER, "peaks")          # per-slice peak fit PNGs
FOLDER_QC       = os.path.join(OUTPUT_FOLDER, "QC_raw")         # per-slice QC PNGs
for _f in [OUTPUT_FOLDER, FOLDER_SUMMARY, FOLDER_POLAR,
            FOLDER_DIFF, FOLDER_PEAKS, FOLDER_QC]:
    os.makedirs(_f, exist_ok=True)

print(f"Data folder      : {DATA_FOLDER}")
print(f"Output folder    : {OUTPUT_FOLDER}")
print(f"  summary/       : CSVs + z-stack-wide PNGs")
print(f"  polar_profile/ : per-slice polar profile PNGs")
print(f"  diff_profile/  : per-slice difference map PNGs")
print(f"  peaks/         : per-slice 1D Peak fit PNGs")
print(f"  QC_raw/        : per-slice QC overlay PNGs")
print(f"Voxel size       : xy={VOXEL_XY_UM} µm  |  z={VOXEL_Z_UM} µm")
print(f"DT channel       : {DT_CHANNEL} → '{CHANNEL_MARKERS[DT_CHANNEL]}'")
print(f"Membrane channel : {MEMBRANE_CH} → '{CHANNEL_MARKERS[MEMBRANE_CH]}'")
print(f"Mask channel     : {'None (disabled)' if MASK_CHANNEL < 0 else str(MASK_CHANNEL) + ' → ' + repr(CHANNEL_MARKERS.get(MASK_CHANNEL, '?'))}")
print(f"Receptor channels: {RECEPTOR_CHANNELS} → {[CHANNEL_MARKERS[c] for c in RECEPTOR_CHANNELS]}")
print(f"Feature size     : {FEATURE_SIZE_UM} µm  (θ step computed after DT load)")
print(f"DT range         : {DT_RANGE_UM[0]} → {DT_RANGE_UM[1]} µm  "
      f"({dt_span_um:.2f} µm span = {dt_span_px:.1f} px)")
print(f"DT bins          : {N_DT_BINS}  "
      f"(n={DT_BINS_PER_PIXEL} × {dt_span_px:.1f} px)  "
      f"bin width = {bin_width_um:.4f} µm = {bin_width_um/VOXEL_XY_UM:.2f} px")

# ── Start run timer and snapshot all user parameters ─────────────────────
import datetime as _dt, time as _time
_RUN_START_TIME  = _time.perf_counter()
_RUN_START_STAMP = _dt.datetime.now()

_CONFIG_SNAPSHOT = {
    "run_date":             _RUN_START_STAMP.strftime("%Y-%m-%d"),
    "run_time":             _RUN_START_STAMP.strftime("%H:%M:%S"),
    "data_folder":          str(Path(DATA_FOLDER).resolve()),
    "output_folder":        str(Path(OUTPUT_FOLDER).resolve()),
    "voxel_xy_um":          VOXEL_XY_UM,
    "voxel_z_um":           VOXEL_Z_UM,
    "channel_markers":      str(CHANNEL_MARKERS),
    "dt_channel":           DT_CHANNEL,
    "membrane_ch":          MEMBRANE_CH,
    "mask_channel":         MASK_CHANNEL,
    "mask_dilation_px":     MASK_DILATION_PX,
    "receptor_channels":    str(RECEPTOR_CHANNELS),
    "channel_range":        str(CHANNEL_RANGE),
    "z_slice_range":        str(Z_SLICE_RANGE),
    "window_size":          WINDOW_SIZE,
    "feature_size_um":      FEATURE_SIZE_UM,
    "dt_range_um":          str(DT_RANGE_UM),
    "dt_bins_per_pixel":    DT_BINS_PER_PIXEL,
    "n_dt_bins":            N_DT_BINS,
    "bin_width_um":         round(bin_width_um, 6),
    "radial_step_px":       RADIAL_STEP_PX,
    "peak_min_intensity": str(PEAK_MIN_INTENSITY),
    "peak_min_sep_um":      PEAK_MIN_SEP_UM,
    "peak_rel_height":      PEAK_REL_HEIGHT,
    "dt_smooth_sigma_px":   DT_SMOOTH_SIGMA_PX,
    "boundary_tol_um":      BOUNDARY_TOL_UM,
    "receptorA_ch":         RECEPTORA_CH,
    "receptorB_ch":         RECEPTORB_CH,
    "n_jobs":               N_JOBS,
}
print(f"\nRun started : {_RUN_START_STAMP.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Timer started — elapsed time will be saved to results/run_log.txt")


---
## 3. File Discovery

In [ ]:
def discover_files(data_folder, channel_range=None, z_slice_range=None):
    """
    Scan data_folder for TIFFs matching *_C{ch}_Z{z}.tif (any zero-padding).
    Returns:
        files       : dict  {ch_int: {z_int: abs_path}}
        sample_name : str   extracted from the first matched filename
    """
    # _C{ch} captures the channel number regardless of leading zeros in channel
    # _Z{z}  captures the z-slice number regardless of zero-padding width
    # int() conversion strips leading zeros in both cases
    pattern = re.compile(
        r"(?P<sample>.+)_C(?P<ch>\d+)_Z(?P<z>\d+)\.tiff?",
        re.IGNORECASE,
    )
    all_tifs = sorted(
        glob.glob(os.path.join(data_folder, "*.tif")) +
        glob.glob(os.path.join(data_folder, "*.tiff"))
    )
    if not all_tifs:
        raise FileNotFoundError(f"No TIFF files found in: {data_folder}")

    files, sample_name = {}, None
    for fpath in all_tifs:
        m = pattern.match(os.path.basename(fpath))
        if not m:
            continue
        ch, z = int(m.group("ch")), int(m.group("z"))
        if sample_name is None:
            sample_name = m.group("sample")
        if channel_range and not (channel_range[0] <= ch <= channel_range[1]):
            continue
        if z_slice_range and not (z_slice_range[0] <= z <= z_slice_range[1]):
            continue
        files.setdefault(ch, {})[z] = fpath

    if not files:
        raise ValueError("No files matched the requested ranges.")
    return files, sample_name


file_index, sample_name = discover_files(DATA_FOLDER, CHANNEL_RANGE, Z_SLICE_RANGE)

all_channels = sorted(file_index.keys())
all_z_slices = sorted({z for d in file_index.values() for z in d})

print(f"Sample name  : {sample_name}")
print(f"Channels     : {all_channels}")
print(f"Z-slices     : {all_z_slices[0]} → {all_z_slices[-1]}  ({len(all_z_slices)} slices)")

---
## 4. Load TIFF Stack

In [ ]:
def load_stack(file_index, channels, z_slices):
    """
    Load TIFFs into a (n_channels, n_z, H, W) float32 array.
    Missing files are filled with zeros and a warning is printed.
    """
    first_path = file_index[channels[0]][z_slices[0]]
    probe = tifffile.imread(first_path)
    H, W  = probe.shape[-2], probe.shape[-1]

    stack = np.zeros((len(channels), len(z_slices), H, W), dtype=np.float32)
    for ci, ch in enumerate(channels):
        for zi, z in enumerate(z_slices):
            path = file_index[ch].get(z)
            if path is None:
                print(f"  [WARNING] Missing ch={ch}, z={z} — filled with zeros")
                continue
            img = tifffile.imread(path).astype(np.float32)
            if img.ndim == 3:
                img = img[0]
            stack[ci, zi] = img
    return stack, list(channels), list(z_slices)


channels_to_load = sorted(set(RECEPTOR_CHANNELS) | {DT_CHANNEL, MEMBRANE_CH}
                           | ({MASK_CHANNEL} if MASK_CHANNEL >= 0 else set()))
if CHANNEL_RANGE:
    channels_to_load = [c for c in channels_to_load
                        if CHANNEL_RANGE[0] <= c <= CHANNEL_RANGE[1]]

# When WINDOW_SIZE > 1, load extra slices beyond Z_SLICE_RANGE so that
# the rolling-window projection fills window edges at boundary slices.
half_win = WINDOW_SIZE // 2
if WINDOW_SIZE > 1 and half_win > 0:
    file_index_ext, _ = discover_files(
        DATA_FOLDER, CHANNEL_RANGE,
        z_slice_range=[all_z_slices[0] - half_win,
                        all_z_slices[-1] + half_win]
    )
    all_z_ext = sorted({z for d in file_index_ext.values() for z in d})
    print(f"Window size      : {WINDOW_SIZE}  (half={half_win})")
    print(f"Extended z range : {all_z_ext[0]} → {all_z_ext[-1]}  "
          f"({len(all_z_ext)} slices)")
    stack_ext, ch_order, z_order_ext = load_stack(
        file_index_ext, channels_to_load, all_z_ext
    )
else:
    stack_ext = z_order_ext = None
    ch_order  = channels_to_load
    print(f"Window size      : 1  (no smoothing)")

print(f"Loading channels : {channels_to_load}")
print(f"User z-slices    : {all_z_slices[0]} → {all_z_slices[-1]}  "
      f"({len(all_z_slices)} slices)")

stack_user, ch_order, z_order_user = load_stack(
    file_index, channels_to_load, all_z_slices
)
print(f"\nUser stack shape (ch, z, y, x): {stack_user.shape}")
if stack_ext is not None:
    print(f"Extended stack shape          : {stack_ext.shape}")

---
## 4b. Rolling-Window Average-Intensity Projection

Each output slice `z` = mean intensity of slices `[z − half, ..., z, ..., z + half]`.
- `WINDOW_SIZE = 1` → original stack, no smoothing
- `WINDOW_SIZE = 3` → each slice averaged with its ±1 neighbours
- `WINDOW_SIZE = 5` → averaged with ±2 neighbours

The **DT channel is never averaged** — it encodes 3D geometry, not fluorescence.
The output `stack` and `z_order` replace the raw versions for all downstream cells.

In [ ]:
def rolling_avg_projection(stack_ext, z_order_ext, z_order_target,
                             dt_channel_idx, window_size,
                             passthrough_idxs=None):
    """
    Apply rolling-window average-intensity projection over z.

    For each z in z_order_target, averages receptor channels over the
    window [z - half, ..., z + half] using the extended stack.
    Channels in passthrough_idxs (e.g. DT, mask) are copied unchanged
    from the exact target z-slice.

    Parameters
    ----------
    stack_ext        : (n_ch, n_z_ext, H, W)  extended raw stack
    z_order_ext      : list  z-slice indices corresponding to stack_ext axis-1
    z_order_target   : list  z-slice indices to produce output for
    dt_channel_idx   : int   axis-0 position of the DT channel
    window_size      : int   must be odd and ≥ 1
    passthrough_idxs : set of axis-0 indices to copy without averaging

    Returns
    -------
    stack_smooth : (n_ch, n_target, H, W) float32
    """
    if passthrough_idxs is None:
        passthrough_idxs = {dt_channel_idx}
    else:
        passthrough_idxs = set(passthrough_idxs) | {dt_channel_idx}

    half      = window_size // 2
    z_ext_arr = np.array(z_order_ext)
    n_ch, _, H, W = stack_ext.shape
    n_target  = len(z_order_target)
    out       = np.zeros((n_ch, n_target, H, W), dtype=np.float32)

    for ti, z_target in enumerate(z_order_target):
        target_ext_idx = np.where(z_ext_arr == z_target)[0][0]
        z_win_vals = [z for z in range(z_target - half, z_target + half + 1)
                      if z in z_order_ext]
        ext_idxs = [np.where(z_ext_arr == z)[0][0] for z in z_win_vals]

        for ci in range(n_ch):
            if ci in passthrough_idxs:
                # Pass through unchanged — geometry/mask channels
                out[ci, ti] = stack_ext[ci, target_ext_idx]
            else:
                out[ci, ti] = stack_ext[ci, ext_idxs].mean(axis=0)

    return out


dt_channel_idx   = channels_to_load.index(DT_CHANNEL)
passthrough_idxs = {dt_channel_idx}
if MASK_CHANNEL >= 0:
    passthrough_idxs.add(channels_to_load.index(MASK_CHANNEL))

if WINDOW_SIZE > 1 and stack_ext is not None:
    stack   = rolling_avg_projection(
        stack_ext, z_order_ext, all_z_slices,
        dt_channel_idx, WINDOW_SIZE,
        passthrough_idxs=passthrough_idxs
    )
    z_order = list(all_z_slices)
    print(f"Rolling projection applied  (window={WINDOW_SIZE}).")
    print(f"Channels smoothed: {[ch for ci,ch in enumerate(channels_to_load) if ci not in passthrough_idxs]}")
    print(f"Channels passed through unchanged: {[ch for ci,ch in enumerate(channels_to_load) if ci in passthrough_idxs]}")
else:
    stack   = stack_user
    z_order = list(z_order_user)
    print("Window size = 1: using original z-stack.")

print(f"\nFinal stack shape (ch, z, y, x): {stack.shape}")
print(f"z_order: {z_order[0]} → {z_order[-1]}  ({len(z_order)} slices)")

---
## 5. Extract Signed DT and Cell Centroids

The DT channel (`C3`) is a **pre-computed signed 3D distance transform** already in **µm**:
- `DT < 0`  → inside the cell
- `DT > 0`  → outside the cell
- `DT ≈ 0`  → cell surface boundary

No recomputation is needed — we use it directly as `all_DT`.

In [ ]:
from scipy.ndimage import gaussian_filter, distance_transform_edt


def extract_dt_and_centroids(stack, ch_order, z_order, dt_channel,
                               smooth_sigma_px=1.0, voxel_xy_um=0.025):
    """
    For each z-slice:
      1. Gaussian-smooth the pre-computed DT channel (C3) with sigma=smooth_sigma_px
         to obtain a clean cell/background boundary.
      2. Threshold the smoothed image at zero to produce a binary cell mask
         (interior = smoothed DT < 0).
      3. Find the largest connected component → cell centroid.
      4. Compute a fresh 2D signed distance transform (EDT) from the binary mask:
             DT_2D = -(dt_inside - dt_outside) * voxel_xy_um
         Convention (matches pre-computed C3): negative = inside, positive = outside.

    The recomputed 2D DT is used for all downstream analysis
    (polar profiling, boundary radius, region masks) instead of the
    pre-computed 3D signed DT.

    Parameters
    ----------
    stack           : (n_ch, n_z, H, W) float32 — full image stack
    ch_order        : list of channel indices
    z_order         : list of z-slice indices
    dt_channel      : int — channel index of the pre-computed DT (used only
                      for boundary detection, not kept in output)
    smooth_sigma_px : float — Gaussian sigma in pixels applied before thresholding
    voxel_xy_um     : float — lateral pixel size in µm

    Returns
    -------
    all_DT_stack : (n_z, H, W) float32 — fresh 2D signed DT in µm
                   negative = cell interior, positive = exterior
    centroids    : list of (cy, cx) tuples in pixels (one per z-slice)
    """
    dt_ci = ch_order.index(dt_channel)
    n_z_  = len(z_order)
    H_    = stack.shape[2]
    W_    = stack.shape[3]
    all_DT_stack = np.zeros((n_z_, H_, W_), dtype=np.float32)
    centroids    = []

    for zi in range(n_z_):
        raw_dt = stack[dt_ci, zi]   # pre-computed DT slice — used for boundary only

        # ── Step 1: smooth and threshold ─────────────────────────────────
        if smooth_sigma_px > 0:
            smoothed = gaussian_filter(raw_dt, sigma=smooth_sigma_px)
        else:
            smoothed = raw_dt
        cell_mask = smoothed < 0   # interior pixels

        # ── Step 2: largest connected component → centroid ────────────────
        labeled_, n_obj = label(cell_mask)
        if n_obj == 0:
            cy, cx = H_ / 2.0, W_ / 2.0
            print(f"  [WARNING] z={z_order[zi]}: no interior found — using image centre")
        else:
            props   = regionprops(labeled_)
            largest = max(props, key=lambda p: p.area)
            cy, cx  = largest.centroid

        centroids.append((cy, cx))

        # ── Step 3: fresh 2D signed EDT ───────────────────────────────────
        dt_in  = distance_transform_edt(cell_mask)    # px from interior to boundary
        dt_out = distance_transform_edt(~cell_mask)   # px from exterior to boundary
        # Signed: negative inside, positive outside (matches C3 convention)
        dt_2d  = -(dt_in - dt_out) * voxel_xy_um      # convert px → µm
        all_DT_stack[zi] = dt_2d

    return all_DT_stack, centroids


all_DT_stack, centroids = extract_dt_and_centroids(
    stack, ch_order, z_order, DT_CHANNEL,
    smooth_sigma_px=DT_SMOOTH_SIGMA_PX,
    voxel_xy_um=VOXEL_XY_UM,
)

n_z = len(z_order)
H, W = all_DT_stack.shape[1], all_DT_stack.shape[2]

print(f"DT stack shape : {all_DT_stack.shape}")
print(f"DT range       : {all_DT_stack.min():.3f} → {all_DT_stack.max():.3f} µm")
print(f"DT source      : 2D EDT recomputed per slice  "
      f"(Gaussian smooth σ={DT_SMOOTH_SIGMA_PX} px applied before thresholding)")
print(f"Convention     : negative = cell interior, positive = exterior")
print(f"Sample centroids (first 5 z-slices):")
for zi in range(min(5, n_z)):
    cy, cx = centroids[zi]
    print(f"  z={z_order[zi]:4d}  centroid=({cx:.1f}, {cy:.1f}) px  "
          f"= ({cx*VOXEL_XY_UM:.2f}, {cy*VOXEL_XY_UM:.2f}) µm")


# ── Measure per-slice boundary radius and derive THETA_ANGLES ────────────
def measure_boundary_radius(dt_slice, cx, cy, H, W, n_probe=360):
    """
    March n_probe evenly-spaced rays from (cx, cy) and find the first
    DT sign change per ray (= cell surface crossing).
    Uses linear interpolation for sub-pixel accuracy.
    Returns an array of crossing distances in pixels
    (rays that reach the image edge without crossing are omitted).
    """
    radii_px = []
    thetas   = np.linspace(0, 2 * np.pi, n_probe, endpoint=False)
    for th in thetas:
        dx, dy   = np.cos(th), np.sin(th)
        prev_val = dt_slice[int(round(cy)), int(round(cx))]
        r = 1.0
        while r <= max(H, W):
            col = int(round(cx + r * dx))
            row = int(round(cy + r * dy))
            if not (0 <= row < H and 0 <= col < W):
                break
            val = dt_slice[row, col]
            if prev_val * val <= 0 and (abs(prev_val) + abs(val)) > 0:
                frac = abs(prev_val) / (abs(prev_val) + abs(val) + 1e-12)
                radii_px.append(r - 1.0 + frac)
                break
            prev_val = val
            r += 1.0
    return np.array(radii_px)


# Average boundary radius across all z-slices
slice_radii_um     = []   # mean boundary radius per z-slice (µm)
slice_n_angles     = []   # n_angles for each z-slice
slice_theta_angles = []   # theta angle array for each z-slice

for zi in range(n_z):
    cy, cx = centroids[zi]
    r_px   = measure_boundary_radius(all_DT_stack[zi], cx, cy, H, W)
    r_um   = float(r_px.mean() * VOXEL_XY_UM) if r_px.size > 0 else np.nan
    slice_radii_um.append(r_um)

    if np.isfinite(r_um) and r_um > 0:
        theta_raw = np.degrees(FEATURE_SIZE_UM / r_um)
        n         = max(1, int(np.ceil(360.0 / theta_raw)))
    else:
        n = 72   # fallback: 5° steps
    slice_n_angles.append(n)
    slice_theta_angles.append(np.linspace(0, 360, n, endpoint=False))

slice_radii_um = np.array(slice_radii_um)

print(f"\nPer-slice boundary radii and θ steps:")
for zi in range(n_z):
    r   = slice_radii_um[zi]
    n   = slice_n_angles[zi]
    arc = r * np.deg2rad(360.0 / n)
    print(f"  z={z_order[zi]:4d}  r={r:.3f} µm  "
          f"n_angles={n:4d}  θ_step={360.0/n:.4f}°  arc={arc:.4f} µm")

print(f"\nFeature size : {FEATURE_SIZE_UM} µm")
print(f"Radius range : {slice_radii_um.min():.3f} – {slice_radii_um.max():.3f} µm")
print(f"n_angles range: {min(slice_n_angles)} – {max(slice_n_angles)}")

---
---
## 5b. Dilate Mask Channel and Compute Polar Theta Mask

If `MASK_CHANNEL >= 0`:
1. **Dilate** the binary mask by `MASK_DILATION_PX` pixels to add a buffer
   around each masked region.
2. For each z-slice and each θ-bin, march a ray from the centroid and check
   whether **any pixel within `DT_RANGE_UM`** along that ray falls inside the
   dilated mask. If so, that θ-bin is flagged as masked.
3. The per-slice boolean array `theta_mask_per_zi[zi]` (True = masked out) is
   stored and passed to all downstream steps.

Masked θ-bins are **excluded** from:
- Raw and normalized polar profile values (set to NaN)
- 1D Peak fitting
- Outside intensity ratio

They are **annotated** on the polar profile and peak plots with a grey shaded
band so the excluded angular range is clearly visible.

In [ ]:
from scipy.ndimage import binary_dilation

# theta_mask_per_zi[zi] is a bool array of length n_theta_zi:
#   True  = this θ-bin is masked out (do not analyze)
#   False = this θ-bin is valid
# When MASK_CHANNEL < 0, all entries are False (nothing masked).
# Populated here; used in Cells 5, 7, 10, 12.
theta_mask_per_zi = []   # list of (n_theta_zi,) bool arrays

if MASK_CHANNEL >= 0:
    mask_ci = ch_order.index(MASK_CHANNEL)
    H_m, W_m = stack.shape[2], stack.shape[3]

    # Dilation structuring element: square of side (2*MASK_DILATION_PX+1)
    if MASK_DILATION_PX > 0:
        _struct = np.ones(
            (2*MASK_DILATION_PX + 1, 2*MASK_DILATION_PX + 1), dtype=bool
        )
    else:
        _struct = None   # no dilation

    print(f"Mask channel     : {MASK_CHANNEL} → "
          f"'{CHANNEL_MARKERS.get(MASK_CHANNEL, '?')}'")
    print(f"Dilation buffer  : {MASK_DILATION_PX} px")

    for zi in range(n_z):
        # NOTE: n_z and slice_theta_angles are defined in Cell 4.
        # Cell 3c must be re-run after Cell 4 if cells are run out of order.
        mask_slice_raw = stack[mask_ci, zi] > 0.5   # (H, W) bool

        # Dilate
        if _struct is not None and mask_slice_raw.any():
            mask_dilated = binary_dilation(mask_slice_raw, structure=_struct)
        else:
            mask_dilated = mask_slice_raw

        # For each θ-bin, march ray and check if any in-range pixel is masked
        cy_zi, cx_zi = centroids[zi]
        theta_z_zi   = slice_theta_angles[zi]
        n_theta_zi   = len(theta_z_zi)
        theta_masked_zi = np.zeros(n_theta_zi, dtype=bool)

        for ti, th_deg in enumerate(theta_z_zi):
            th = np.deg2rad(th_deg)
            dx, dy = np.cos(th), np.sin(th)
            r = 0.0
            while r <= max(H_m, W_m):
                col  = int(round(cx_zi + r * dx))
                row_ = int(round(cy_zi + r * dy))
                if not (0 <= row_ < H_m and 0 <= col < W_m):
                    break
                dt_val = all_DT_stack[zi, row_, col]
                if DT_RANGE_UM[0] <= dt_val <= DT_RANGE_UM[1]:
                    if mask_dilated[row_, col]:
                        theta_masked_zi[ti] = True
                        break
                r += 1

        theta_mask_per_zi.append(theta_masked_zi)

        n_m = int(theta_masked_zi.sum())
        if (zi + 1) % 10 == 0 or zi == n_z - 1:
            print(f"  z={z_order[zi]:4d}  "
                  f"mask px={mask_dilated.sum():6d}  "
                  f"θ-bins masked={n_m}/{n_theta_zi}  "
                  f"({100*n_m/n_theta_zi:.1f}%)")

    print(f"\nTheta mask computed for all {n_z} z-slices.")

else:
    # No mask channel — create all-False arrays (nothing excluded)
    for zi in range(n_z):
        theta_mask_per_zi.append(
            np.zeros(len(slice_theta_angles[zi]), dtype=bool)
        )
    print("MASK_CHANNEL = -1: no theta masking applied.")

---
## 6. Polar-Coordinate Receptor Profiling

In [ ]:
def compute_polar_profile(receptor_slice, dt_slice, centroid,
                           theta_angles, bin_edges_um, H, W,
                           radial_step_px=1):
    """
    For each angle in theta_angles, march a ray from centroid outward
    and collect receptor intensities binned by DT value (µm).

    Returns
    -------
    profile : (n_theta, n_bins) float32 — NaN where no pixels fell in a bin
    """
    cy, cx = centroid
    n_bins = len(bin_edges_um) - 1
    profile = np.full((len(theta_angles), n_bins), np.nan, dtype=np.float32)
    dt_min, dt_max = bin_edges_um[0], bin_edges_um[-1]

    for ti, th_deg in enumerate(theta_angles):
        th = np.deg2rad(th_deg)
        dx, dy = np.cos(th), np.sin(th)
        ray_dt, ray_int = [], []
        r = 0.0
        while r <= max(H, W):
            col = int(round(cx + r * dx))
            row = int(round(cy + r * dy))
            if not (0 <= row < H and 0 <= col < W):
                break
            dt_val = dt_slice[row, col]
            if dt_min <= dt_val <= dt_max:
                ray_dt.append(dt_val)
                ray_int.append(receptor_slice[row, col])
            r += radial_step_px
        if not ray_dt:
            continue
        ray_dt  = np.array(ray_dt)
        ray_int = np.array(ray_int)
        for bi in range(n_bins):
            m = (ray_dt >= bin_edges_um[bi]) & (ray_dt < bin_edges_um[bi + 1])
            if m.any():
                profile[ti, bi] = ray_int[m].mean()
    return profile


def normalize_profile(profile):
    """
    Min-max normalize a (n_theta, n_bins) profile to [0, 1].
    Only non-NaN, non-zero values contribute to min/max.
    NaN and zero positions are preserved.
    """
    valid_mask = (~np.isnan(profile)) & (profile != 0.0)
    valid = profile[valid_mask]
    if valid.size == 0:
        return profile.copy().astype(np.float32)
    vmin, vmax = valid.min(), valid.max()
    denom = vmax - vmin
    if denom < 1e-8:
        out = np.where(valid_mask, 0.5, profile)
        return np.where(np.isnan(profile), np.nan, out).astype(np.float32)
    out = np.where(valid_mask, (profile - vmin) / denom, profile)
    return np.where(np.isnan(profile), np.nan, out).astype(np.float32)


def interpolate_to_common_grid(profile, src_theta, dst_theta):
    """
    Linearly interpolate a (n_src_theta, n_bins) profile onto dst_theta.
    Wraps around 360 degrees. Returns (n_dst_theta, n_bins) float32.
    """
    from scipy.interpolate import interp1d
    n_bins = profile.shape[1]
    out    = np.full((len(dst_theta), n_bins), np.nan, dtype=np.float32)
    src_ext = np.concatenate([src_theta - 360, src_theta, src_theta + 360])
    for bi in range(n_bins):
        col_ext = np.tile(profile[:, bi], 3)
        valid   = ~np.isnan(col_ext)
        if valid.sum() < 2:
            continue
        f = interp1d(src_ext[valid], col_ext[valid],
                     kind="linear", bounds_error=False, fill_value=np.nan)
        out[:, bi] = f(dst_theta).astype(np.float32)
    return out


# --- Run for all receptor channels x all z-slices ---
# results_unmasked[ch] : raw profile, ALL theta-bins (no masking)
# results[ch]          : raw profile, masked theta-bins set to NaN
# results_norm[ch]     : normalized masked profile
results_unmasked = {ch: [] for ch in RECEPTOR_CHANNELS}
results          = {ch: [] for ch in RECEPTOR_CHANNELS}
results_norm     = {ch: [] for ch in RECEPTOR_CHANNELS}

print(f"Computing polar profiles in parallel (N_JOBS={N_JOBS})...")


def _compute_one_slice_ch(zi, ch):
    """Compute unmasked raw, masked raw, and normalised profile for (zi, ch)."""
    ci    = ch_order.index(ch)
    theta = slice_theta_angles[zi]
    prof  = compute_polar_profile(
        stack[ci, zi], all_DT_stack[zi], centroids[zi],
        theta, BIN_EDGES_UM, H, W, RADIAL_STEP_PX,
    )
    prof_unmasked = prof.copy()                         # all theta-bins present
    prof[theta_mask_per_zi[zi], :] = np.nan             # apply theta mask
    return zi, ch, prof_unmasked, prof, normalize_profile(prof)


tasks = [(zi, ch) for zi in range(n_z) for ch in RECEPTOR_CHANNELS]

raw_results = Parallel(n_jobs=N_JOBS, prefer="threads", verbose=0)(
    delayed(_compute_one_slice_ch)(zi, ch) for zi, ch in tasks
)

_buf_u    = {ch: {} for ch in RECEPTOR_CHANNELS}
_buf      = {ch: {} for ch in RECEPTOR_CHANNELS}
_buf_norm = {ch: {} for ch in RECEPTOR_CHANNELS}
for zi, ch, prof_u, prof, prof_norm in raw_results:
    _buf_u[ch][zi]    = prof_u
    _buf[ch][zi]      = prof
    _buf_norm[ch][zi] = prof_norm
for ch in RECEPTOR_CHANNELS:
    results_unmasked[ch] = [_buf_u[ch][zi]    for zi in range(n_z)]
    results[ch]          = [_buf[ch][zi]      for zi in range(n_z)]
    results_norm[ch]     = [_buf_norm[ch][zi] for zi in range(n_z)]

print(f"\nDone. Profiles computed for {n_z} z-slices x {len(RECEPTOR_CHANNELS)} channels.")
print(f"n_bins = {len(DT_BINS_UM)}")
for zi in range(min(3, n_z)):
    ch0 = RECEPTOR_CHANNELS[0]
    n_m = int(theta_mask_per_zi[zi].sum())
    print(f"  z={z_order[zi]:4d}: shape={results[ch0][zi].shape}  "
          f"\u03b8-masked={n_m}/{len(slice_theta_angles[zi])}")

---
## 7. Export Results to CSV

In [ ]:
def export_csv(results, results_norm, slice_theta_angles, dt_bins_um, z_order,
               channel_markers, receptor_channels,
               centroids, slice_radii_um, output_folder, sample_name):
    """Save long-format CSV per receptor channel (raw + normalized) + centroid table.
    Each row carries the theta_deg value from that slice's own angle grid."""

    for ch in receptor_channels:
        marker = channel_markers[ch]
        rows   = []
        for zi in range(len(z_order)):
            prof  = results[ch][zi]       # (n_theta_z, n_bins)
            norm  = results_norm[ch][zi]  # (n_theta_z, n_bins)
            theta_z = slice_theta_angles[zi]
            for ti, theta in enumerate(theta_z):
                for bi, dt_val in enumerate(dt_bins_um):
                    rows.append({
                        "z_slice": z_order[zi],
                        "theta_deg": round(float(theta), 6),
                        "dt_um": round(float(dt_val), 4),
                        "mean_intensity": float(prof[ti, bi]),
                        "mean_intensity_norm": float(norm[ti, bi]),
                    })
        path = os.path.join(output_folder, "summary",
                            f"{sample_name}_{marker}_polar_profile.csv")
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"  Saved: {path}")

    # Centroid table (includes per-slice boundary radius and n_angles)
    cent_df = pd.DataFrame([
        {"z_slice": z_order[zi],
         "centroid_row_px": centroids[zi][0],
         "centroid_col_px": centroids[zi][1],
         "centroid_x_um":  centroids[zi][1] * VOXEL_XY_UM,
         "centroid_y_um":  centroids[zi][0] * VOXEL_XY_UM,
         "boundary_radius_um": float(slice_radii_um[zi]),
         "n_theta": len(slice_theta_angles[zi]),
         "theta_step_deg": round(360.0 / len(slice_theta_angles[zi]), 6)}
        for zi in range(len(z_order))
    ])
    cent_path = os.path.join(output_folder, "summary", f"{sample_name}_centroids.csv")
    cent_df.to_csv(cent_path, index=False)
    print(f"  Saved: {cent_path}")


export_csv(results, results_norm, slice_theta_angles, DT_BINS_UM, z_order,
           CHANNEL_MARKERS, RECEPTOR_CHANNELS,
           centroids, slice_radii_um, OUTPUT_FOLDER, sample_name)
print("CSV export complete.")

---
## 8. Per-Slice Polar Profile Plots

In [ ]:
def plot_slice(zi, results_unmasked, results, results_norm,
               slice_theta_angles, dt_bins_um,
               receptor_channels, channel_markers,
               z_label, output_folder, sample_name,
               channel_cmaps, theta_mask_list=None, save=True):
    """
    Three-column layout per receptor channel:
      Col 0: Unmasked raw     — all theta-bins, no masking applied
      Col 1: Masked raw       — masked theta-bins shown as grey bands
      Col 2: Masked normalized — same masking, normalized [0, 1]
    """
    n_ch    = len(receptor_channels)
    theta_z = slice_theta_angles[zi]
    t_mask  = (theta_mask_list[zi] if theta_mask_list is not None
               else np.zeros(len(theta_z), dtype=bool))
    n_theta    = len(theta_z)
    theta_step = 360.0 / n_theta
    n_masked   = int(t_mask.sum())

    fig, axes = plt.subplots(
        n_ch, 3,
        figsize=(22, 3.8 * n_ch),
        sharex=True, sharey=True,
        constrained_layout=True
    )
    if n_ch == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        f"{sample_name}  |  z-slice {z_label}  |"
        f"  n_\u03b8={n_theta}  \u03b8_step={theta_step:.3f}\u00b0"
        f"  |  DT {dt_bins_um[0]:.1f}\u2192{dt_bins_um[-1]:.1f} \u00b5m"
        + (f"  |  \u03b8-masked={n_masked}" if n_masked > 0 else ""),
        fontsize=11, fontweight="bold"
    )

    def _shade(ax):
        """Shade contiguous runs of masked theta-bins with grey."""
        if not t_mask.any():
            return
        in_span = False
        span_start = None
        for ti in range(n_theta):
            if t_mask[ti] and not in_span:
                span_start = theta_z[ti] - theta_step / 2
                in_span = True
            elif not t_mask[ti] and in_span:
                ax.axvspan(span_start, theta_z[ti - 1] + theta_step / 2,
                           color="grey", alpha=0.5, linewidth=0, zorder=3)
                in_span = False
        if in_span:
            ax.axvspan(span_start, theta_z[-1] + theta_step / 2,
                       color="grey", alpha=0.5, linewidth=0, zorder=3)
        ax.plot([], [], color="grey", alpha=0.6, lw=6, label="masked")

    for row_i, ch in enumerate(receptor_channels):
        marker     = channel_markers[ch]
        cmap       = channel_cmaps[row_i % len(channel_cmaps)]
        p_unmasked = results_unmasked[ch][zi]   # (n_theta, n_bins) all bins
        p_masked   = results[ch][zi]             # (n_theta, n_bins) NaN on masked
        p_norm     = results_norm[ch][zi]        # (n_theta, n_bins) normalized

        # ── Col 0: Unmasked raw ───────────────────────────────────────────
        im0 = axes[row_i, 0].pcolormesh(
            theta_z, dt_bins_um, p_unmasked.T, cmap=cmap, shading="auto")
        axes[row_i, 0].axhline(0, color="white", lw=1.0, ls="--", label="boundary")
        axes[row_i, 0].set_title(
            f"Ch {ch}: {marker} \u2014 Unmasked raw", fontsize=9, loc="left")
        axes[row_i, 0].set_ylabel(
            "Distance (\u00b5m)\n\u2190 inside | outside \u2192", fontsize=8)
        axes[row_i, 0].legend(fontsize=7, loc="upper right")
        fig.colorbar(im0, ax=axes[row_i, 0], pad=0.01, fraction=0.02
                     ).set_label("Mean intensity", fontsize=7)

        # ── Col 1: Masked raw ─────────────────────────────────────────────
        im1 = axes[row_i, 1].pcolormesh(
            theta_z, dt_bins_um, p_masked.T, cmap=cmap, shading="auto")
        axes[row_i, 1].axhline(0, color="white", lw=1.0, ls="--", label="boundary")
        _shade(axes[row_i, 1])
        axes[row_i, 1].set_title(
            f"Ch {ch}: {marker} \u2014 Masked raw", fontsize=9, loc="left")
        axes[row_i, 1].legend(fontsize=7, loc="upper right")
        fig.colorbar(im1, ax=axes[row_i, 1], pad=0.01, fraction=0.02
                     ).set_label("Mean intensity", fontsize=7)

        # ── Col 2: Masked normalized ──────────────────────────────────────
        im2 = axes[row_i, 2].pcolormesh(
            theta_z, dt_bins_um, p_norm.T, cmap=cmap, shading="auto",
            vmin=0, vmax=1)
        axes[row_i, 2].axhline(0, color="white", lw=1.0, ls="--", label="boundary")
        _shade(axes[row_i, 2])
        axes[row_i, 2].set_title(
            f"Ch {ch}: {marker} \u2014 Masked normalized [0,1]", fontsize=9, loc="left")
        axes[row_i, 2].legend(fontsize=7, loc="upper right")
        fig.colorbar(im2, ax=axes[row_i, 2], pad=0.01, fraction=0.02
                     ).set_label("Norm. intensity", fontsize=7)

    for col in range(3):
        axes[-1, col].set_xlabel("Angle \u03b8 (degrees)", fontsize=9)
        axes[-1, col].set_xlim(theta_z[0], theta_z[-1])

    if save:
        path = os.path.join(output_folder, "polar_profile",
                            f"{sample_name}_z{z_label:04d}_polar_profile.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()


print(f"Generating per-slice plots in parallel (N_JOBS={N_JOBS})...")
Parallel(n_jobs=N_JOBS, prefer="threads", verbose=0)(
    delayed(plot_slice)(
        zi, results_unmasked, results, results_norm,
        slice_theta_angles, DT_BINS_UM,
        RECEPTOR_CHANNELS, CHANNEL_MARKERS,
        z_label=z_order[zi],
        output_folder=OUTPUT_FOLDER,
        sample_name=sample_name,
        channel_cmaps=CHANNEL_CMAPS,
        theta_mask_list=theta_mask_per_zi,
        save=True
    )
    for zi in range(n_z)
)
print(f"All per-slice polar plots saved to: {FOLDER_POLAR}")

---
## 9. Z-Averaged Summary Plot
All receptor channels averaged across z, displayed side-by-side.

In [ ]:
def plot_zavg_summary(results, results_norm, slice_theta_angles, dt_bins_um,
                      receptor_channels, channel_markers,
                      output_folder, sample_name, channel_cmaps):
    """
    Z-averaged polar profiles for all receptor channels.
    Because each z-slice has its own theta grid, all slices are first
    interpolated onto the finest grid (max n_angles) before averaging.
    Two rows: top = raw z-average, bottom = normalized z-average.
    """
    # Common grid = finest theta resolution across all slices
    n_angles_max  = max(len(t) for t in slice_theta_angles)
    common_theta  = np.linspace(0, 360, n_angles_max, endpoint=False)

    n_ch = len(receptor_channels)
    fig, axes = plt.subplots(2, n_ch,
                              figsize=(6 * n_ch, 9),
                              sharex=True, sharey=True,
                              constrained_layout=True)
    if n_ch == 1:
        axes = axes[:, np.newaxis]

    fig.suptitle(
        f"{sample_name} \u2014 Z-averaged polar receptor profiles\n"
        f"(interpolated to common \u03b8 grid: n={n_angles_max}, "
        f"step={360.0/n_angles_max:.3f}\u00b0)",
        fontsize=11, fontweight="bold"
    )

    row_labels = ["Raw (z-mean)", "Normalized (z-mean of norm.)"]
    result_sets = [results, results_norm]
    cb_labels   = ["Mean intensity", "Norm. intensity"]
    vmin_sets   = [None, 0]
    vmax_sets   = [None, 1]

    for row_i, (res, row_lbl, cb_lbl, vmin, vmax) in enumerate(
            zip(result_sets, row_labels, cb_labels, vmin_sets, vmax_sets)):
        for col_i, ch in enumerate(receptor_channels):
            marker = channel_markers[ch]
            cmap   = channel_cmaps[col_i % len(channel_cmaps)]

            # Interpolate each slice to common_theta then nanmean
            interp_stack = np.stack([
                interpolate_to_common_grid(
                    res[ch][zi], slice_theta_angles[zi], common_theta
                )
                for zi in range(len(slice_theta_angles))
            ], axis=0)   # (n_z, n_angles_max, n_bins)
            zavg = np.nanmean(interp_stack, axis=0)  # (n_angles_max, n_bins)

            im = axes[row_i, col_i].pcolormesh(
                common_theta, dt_bins_um, zavg.T,
                cmap=cmap, shading="auto", vmin=vmin, vmax=vmax
            )
            axes[row_i, col_i].axhline(0, color="white", lw=1.2, ls="--")
            axes[row_i, col_i].set_title(
                f"Ch {ch}: {marker}\n{row_lbl}", fontsize=10
            )
            if col_i == 0:
                axes[row_i, col_i].set_ylabel(
                    "Distance (\u00b5m)\n\u2190 inside | outside \u2192", fontsize=9
                )
            if row_i == 1:
                axes[row_i, col_i].set_xlabel("Angle \u03b8 (degrees)", fontsize=9)
            cb = fig.colorbar(im, ax=axes[row_i, col_i], pad=0.02, fraction=0.046)
            cb.set_label(cb_lbl, fontsize=8)

    path = os.path.join(output_folder, "summary", f"{sample_name}_zavg_polar_profile.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Summary plot saved: {path}")


plot_zavg_summary(
    results, results_norm, slice_theta_angles, DT_BINS_UM,
    RECEPTOR_CHANNELS, CHANNEL_MARKERS,
    OUTPUT_FOLDER, sample_name, CHANNEL_CMAPS
)

---
## 10. Peak Detection on Normalized Polar Profiles

For each channel and each z-slice, detect intensity peaks along the **Distance**
(DT) axis of each θ-bin in the normalized polar profile.

**Algorithm per θ-bin:**
1. Find all peaks above `PEAK_ABS_THRESH` with minimum separation `PEAK_MIN_SEP_UM`
   using `scipy.signal.find_peaks`.
2. Keep only peaks whose height ≥ `PEAK_REL_HEIGHT × (height of tallest peak)`,
   discarding minor satellite peaks.
3. Compute the **FWHM** of each selected peak via `scipy.signal.peak_widths`.
   The **global DT range** spans from the left FWHM bound of the leftmost peak
   to the right FWHM bound of the rightmost peak.
4. The **centre** = mean DT of all selected peak positions.
   

Classification of each θ-bin's centre:
- **boundary** : `|centre| < BOUNDARY_TOL_UM`
- **outside**  : `centre ≥ +BOUNDARY_TOL_UM`
- **inside**   : `centre ≤ −BOUNDARY_TOL_UM`

θ-bins in `theta_mask_per_zi` are skipped entirely.

In [ ]:
from scipy.signal import find_peaks, peak_widths
import collections


def _classify(dt_um, tol):
    if abs(dt_um) < tol:
        return "boundary"
    return "outside" if dt_um > 0 else "inside"


def detect_peaks_1d(row, dt_bins_um,
                     abs_thresh, rel_height_ratio, min_sep_um):
    """
    Detect intensity peaks along a single DT profile row (one θ-bin).

    Parameters
    ----------
    row             : 1-D float array — normalized intensity vs DT
    dt_bins_um      : 1-D float array — DT bin centres (µm), same length as row
    abs_thresh      : float — minimum absolute peak height (e.g. 0.1 on [0,1])
    rel_height_ratio: float — keep peaks >= ratio × max_peak_height (e.g. 0.30)
    min_sep_um      : float — minimum separation between peaks (µm)

    Returns
    -------
    dict with keys:
      centre_um   : float — mean DT of selected peak positions (µm)
      range_lo_um : float — left FWHM bound of leftmost selected peak (µm)
      range_hi_um : float — right FWHM bound of rightmost selected peak (µm)
      n_peaks     : int   — number of selected peaks
      peak_dts    : list  — DT positions of all selected peaks (µm)
    or None if no qualifying peak found.
    """
    bin_width    = float(dt_bins_um[1] - dt_bins_um[0]) if len(dt_bins_um) > 1 else 1.0
    min_sep_bins = max(1, int(round(min_sep_um / bin_width)))

    # Replace NaN with 0 for peak finding
    row_clean = np.where(np.isnan(row), 0.0, row)

    # Step 1: find all peaks above absolute threshold
    peak_idxs, _ = find_peaks(row_clean,
                               height=abs_thresh,
                               distance=min_sep_bins)
    if len(peak_idxs) == 0:
        return None

    # Step 2: keep peaks >= rel_height_ratio × tallest peak
    peak_heights = row_clean[peak_idxs]
    major_h      = peak_heights.max()
    sel_mask     = peak_heights >= rel_height_ratio * major_h
    selected     = peak_idxs[sel_mask]
    if len(selected) == 0:
        return None

    # Step 3: FWHM for each selected peak → global DT range
    _, _, left_ips, right_ips = peak_widths(row_clean, selected, rel_height=0.5)
    idx_arr   = np.arange(len(dt_bins_um), dtype=float)
    left_um   = np.interp(left_ips,  idx_arr, dt_bins_um)
    right_um  = np.interp(right_ips, idx_arr, dt_bins_um)

    range_lo  = float(left_um.min())
    range_hi  = float(right_um.max())

    # Step 4: centre = mean of selected peak DT positions
    peak_dts  = dt_bins_um[selected].tolist()
    centre_um = float(np.mean(peak_dts))

    return {
        "centre_um":   centre_um,
        "range_lo_um": range_lo,
        "range_hi_um": range_hi,
        "n_peaks":     int(len(selected)),
        "peak_dts":    peak_dts,
    }


def run_peak_detection(results_norm, slice_theta_angles, dt_bins_um,
                        z_order, channel_markers, receptor_channels,
                        abs_thresh_per_ch, boundary_tol_um,
                        rel_height_ratio, min_sep_um,
                        output_folder, sample_name, channel_cmaps,
                        membrane_ch, theta_mask_list=None):
    """
    Run peak detection for all channels × all z-slices.

    Per-slice figure (2 columns per channel):
      Left  : channel's normalized profile + detected centre markers
      Right : membrane-derived region overlay (greyscale background)

    CSV columns per row:
      z_slice, channel, marker, theta_idx, theta_deg,
      centre_um, range_lo_um, range_hi_um, n_peaks, location
    """
    all_rows = []
    n_ch     = len(receptor_channels)

    LOC_COLOR = {"outside": "cyan", "boundary": "dodgerblue", "inside": "red"}
    LOC_FILL  = {"outside": "cyan", "boundary": "dodgerblue", "inside": "red"}
    LOC_ALPHA = {"outside": 0.6,   "boundary": 0.5,          "inside": 0.6}

    for zi in range(len(z_order)):
        theta_z    = slice_theta_angles[zi]
        n_theta    = len(theta_z)
        theta_step = 360.0 / n_theta
        t_mask     = (theta_mask_list[zi]
                      if theta_mask_list is not None
                      else np.zeros(n_theta, dtype=bool))

        # ── Membrane peaks (once per slice) — define region overlay ──────
        mem_thresh    = abs_thresh_per_ch.get(membrane_ch, 0.1)
        mem_prof      = results_norm[membrane_ch][zi]   # (n_theta, n_bins)
        membrane_hits = {}   # theta_idx → detect_peaks_1d result
        for ti, theta in enumerate(theta_z):
            if t_mask[ti]:
                continue
            row = np.where(np.isnan(mem_prof[ti]), 0.0, mem_prof[ti])
            result = detect_peaks_1d(row, dt_bins_um,
                                      mem_thresh, rel_height_ratio, min_sep_um)
            if result is not None:
                result["location"] = _classify(result["centre_um"], boundary_tol_um)
                membrane_hits[ti]  = result

        # Build membrane_fits list 
        membrane_fits = [
            {"theta_idx": ti, "theta_deg": float(theta_z[ti]),
             "dt_um":       h["centre_um"],
             "range_lo_um": h["range_lo_um"],
             "range_hi_um": h["range_hi_um"],
             "n_peaks":     h["n_peaks"],
             "location":    h["location"]}
            for ti, h in sorted(membrane_hits.items())
        ]

        # ── Figure ────────────────────────────────────────────────────────
        fig, axes = plt.subplots(
            n_ch, 2, figsize=(12, 4 * n_ch),
            sharey=True, constrained_layout=True
        )
        if n_ch == 1:
            axes = axes[np.newaxis, :]
        fig.suptitle(
            f"{sample_name}  |  z={z_order[zi]}  |  Peak detection"
            f"  |  regions from Ch {membrane_ch}: "
            f"{channel_markers[membrane_ch]}",
            fontsize=10, fontweight="bold"
        )

        for row_i, ch in enumerate(receptor_channels):
            marker = channel_markers[ch]
            cmap   = channel_cmaps[row_i % len(channel_cmaps)]
            thresh = abs_thresh_per_ch.get(ch, 0.1)
            prof   = results_norm[ch][zi]

            # Detect peaks for this channel
            ch_hits = {}
            for ti, theta in enumerate(theta_z):
                if t_mask[ti]:
                    continue
                row = np.where(np.isnan(prof[ti]), 0.0, prof[ti])
                result = detect_peaks_1d(row, dt_bins_um,
                                          thresh, rel_height_ratio, min_sep_um)
                if result is not None:
                    result["location"] = _classify(result["centre_um"],
                                                   boundary_tol_um)
                    ch_hits[ti] = result

            for ti, h in ch_hits.items():
                all_rows.append({
                    "z_slice":     z_order[zi],
                    "channel":     ch,
                    "marker":      marker,
                    "theta_idx":   ti,
                    "theta_deg":   float(theta_z[ti]),
                    "centre_um":   h["centre_um"],
                    "range_lo_um": h["range_lo_um"],
                    "range_hi_um": h["range_hi_um"],
                    "n_peaks":     h["n_peaks"],
                    "location":    h["location"],
                })

            # ── Left panel: normalized profile + centre markers ───────────
            ax_fit = axes[row_i, 0]
            ax_fit.pcolormesh(theta_z, dt_bins_um, prof.T,
                              cmap=cmap, shading="auto", vmin=0, vmax=1)
            ax_fit.axhline(0, color="k", lw=0.8, ls="--")
            ax_fit.set_title(
                f"Ch {ch}: {marker}\n"
                f"thresh={thresh:.2f}  detected={len(ch_hits)}",
                fontsize=9, loc="left")
            ax_fit.set_xlabel("\u03b8 (degrees)", fontsize=8)
            ax_fit.set_ylabel(
                "Distance (\u00b5m)\n\u2190 inside | outside \u2192",
                fontsize=8)
            ax_fit.set_xlim(theta_z[0], theta_z[-1])
            for ti, h in ch_hits.items():
                color = LOC_COLOR.get(h["location"], "yellow")
                ax_fit.plot(theta_z[ti], h["centre_um"], "+",
                            color=color, markersize=10, markeredgewidth=1.5)
                ax_fit.annotate(
                    h["location"][0].upper(),
                    (theta_z[ti], h["centre_um"]),
                    textcoords="offset points", xytext=(4, 3),
                    fontsize=6, color=color)

            # ── Right panel: greyscale + MEMBRANE region overlay ──────────
            ax_reg = axes[row_i, 1]
            ax_reg.pcolormesh(theta_z, dt_bins_um, prof.T,
                              cmap="Greys", shading="auto", vmin=0, vmax=1)
            ax_reg.axhline(0, color="k", lw=0.8, ls="--")
            for f in membrane_fits:
                x_lo  = f["theta_deg"] - theta_step / 2
                x_hi  = f["theta_deg"] + theta_step / 2
                y_lo  = max(f["range_lo_um"], dt_bins_um[0])
                y_hi  = min(f["range_hi_um"], dt_bins_um[-1])
                color = LOC_FILL.get(f["location"], "yellow")
                alpha = LOC_ALPHA.get(f["location"], 0.5)
                ax_reg.fill_between(
                    [x_lo, x_hi], y_lo, y_hi,
                    color=color, alpha=alpha, linewidth=0)
            ax_reg.set_title(
                f"Ch {ch}: {marker} — membrane regions\n"
                f"outside=cyan  boundary=blue  inside=red",
                fontsize=9, loc="left")
            ax_reg.set_xlabel("\u03b8 (degrees)", fontsize=8)
            ax_reg.set_xlim(theta_z[0], theta_z[-1])

            # Shade masked theta-bins on both panels
            if t_mask.any():
                in_span = False
                ss = None
                for _ti in range(n_theta):
                    if t_mask[_ti] and not in_span:
                        ss = theta_z[_ti] - theta_step / 2
                        in_span = True
                    elif not t_mask[_ti] and in_span:
                        for _ax in [ax_fit, ax_reg]:
                            _ax.axvspan(ss, theta_z[_ti-1] + theta_step / 2,
                                        color="grey", alpha=0.4, lw=0, zorder=3)
                        in_span = False
                if in_span:
                    for _ax in [ax_fit, ax_reg]:
                        _ax.axvspan(ss, theta_z[-1] + theta_step / 2,
                                    color="grey", alpha=0.4, lw=0, zorder=3)
                ax_fit.plot([], [], color="grey", alpha=0.5, lw=6,
                            label="masked")
                ax_fit.legend(fontsize=6)

        path = os.path.join(output_folder, "peaks",
                            f"{sample_name}_z{z_order[zi]:04d}_peaks.png")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        if (zi + 1) % 10 == 0 or zi == len(z_order) - 1:
            n_det = sum(1 for r in all_rows if r["z_slice"] == z_order[zi])
            print(f"  z={z_order[zi]:4d}  total detections: {n_det}  "
                  f"({zi+1}/{len(z_order)})")

    # Save CSV
    csv_path = os.path.join(output_folder, "summary",
                             f"{sample_name}_peak_fits.csv")
    pd.DataFrame(all_rows).to_csv(csv_path, index=False)
    print(f"\nPeak detection CSV saved: {csv_path}")
    print(f"Total detections        : {len(all_rows)}")

    for ch in receptor_channels:
        subset = [r for r in all_rows if r["channel"] == ch]
        counts = collections.Counter(r["location"] for r in subset)
        thresh = abs_thresh_per_ch.get(ch, 0.1)
        print(f"  Ch {ch} ({channel_markers[ch]})  "
              f"abs_thresh={thresh:.2f}:  {dict(counts)}")

    return all_rows


# ── Run ───────────────────────────────────────────────────────────────────
print("Running peak detection...")
print(f"  PEAK_MIN_INTENSITY: {PEAK_MIN_INTENSITY}")
print(f"  rel_height_ratio  : {PEAK_REL_HEIGHT}")
print(f"  min_sep_um        : {PEAK_MIN_SEP_UM} \u00b5m")
print(f"  boundary_tol_um   : \u00b1{BOUNDARY_TOL_UM} \u00b5m")

peak_fit_rows = run_peak_detection(
    results_norm,
    slice_theta_angles, DT_BINS_UM,
    z_order, CHANNEL_MARKERS, RECEPTOR_CHANNELS,
    abs_thresh_per_ch=PEAK_MIN_INTENSITY,
    boundary_tol_um=BOUNDARY_TOL_UM,
    rel_height_ratio=PEAK_REL_HEIGHT,
    min_sep_um=PEAK_MIN_SEP_UM,
    output_folder=OUTPUT_FOLDER,
    sample_name=sample_name,
    channel_cmaps=CHANNEL_CMAPS,
    membrane_ch=MEMBRANE_CH,
    theta_mask_list=theta_mask_per_zi,
)

---
## 11. QC: Raw Channels + DT Overlay for One Z-slice

In [ ]:
def peak_to_pixel(theta_deg, dt_um, cx, cy, dt_slice, H, W, step=0.5):
    """
    March along the ray at theta_deg from centroid (cx, cy).
    Returns the (col, row) pixel whose DT value is closest to dt_um.
    """
    th = np.deg2rad(theta_deg)
    dx, dy = np.cos(th), np.sin(th)
    best_px, best_diff = None, np.inf
    r = 0.0
    while r <= max(H, W):
        col = int(round(cx + r * dx))
        row = int(round(cy + r * dy))
        if not (0 <= row < H and 0 <= col < W):
            break
        diff = abs(dt_slice[row, col] - dt_um)
        if diff < best_diff:
            best_diff = diff
            best_px = (col, row)
        if dt_slice[row, col] > dt_um + 1.0:   # well past target
            break
        r += step
    return best_px


def plot_qc_slice(zi, stack, all_DT_stack, centroids, z_order,
                   receptor_channels, channel_markers, dt_channel,
                   peak_fit_rows, H, W, output_folder, sample_name):
    """
    QC overlay for one z-slice: raw receptor images + signed DT panel
    with cell boundary contour, centroid, θ reference rays, and
    outside-peak markers from peak fitting.
    Saved to output_folder/QC_raw/.
    """
    cy, cx      = centroids[zi]
    dt_slice_qc = all_DT_stack[zi]
    z_label_qc  = z_order[zi]

    outside_peaks = [
        r for r in peak_fit_rows
        if r["z_slice"] == z_label_qc and r["location"] == "outside"
    ]

    n_ch   = len(receptor_channels)
    n_cols = n_ch + 1
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5),
                              constrained_layout=True)
    fig.suptitle(
        f"{sample_name}  |  z={z_label_qc}  |  QC overview  "
        f"(\u25cb = outside peaks)",
        fontsize=11, fontweight="bold"
    )

    FIT_MARKER = "o"
    CH_COLORS  = ["lime", "yellow", "magenta", "cyan", "orange"]

    for ax_i, ch in enumerate(receptor_channels):
        ci  = ch_order.index(ch)
        img = stack[ci, zi]
        im  = axes[ax_i].imshow(img, cmap="hot", origin="upper")
        axes[ax_i].contour(dt_slice_qc, levels=[0], colors="cyan", linewidths=0.8)
        axes[ax_i].plot(cx, cy, "c+", markersize=10, markeredgewidth=1.5)
        axes[ax_i].set_title(f"Ch {ch}: {channel_markers[ch]}")
        fig.colorbar(im, ax=axes[ax_i], fraction=0.046, pad=0.04)

        ch_peaks  = [r for r in outside_peaks if r["channel"] == ch]
        seen_lbls = set()
        for pk in ch_peaks:
            px = peak_to_pixel(pk["theta_deg"], pk["centre_um"],
                               cx, cy, dt_slice_qc, H, W)
            if px is None:
                continue
            col_px, row_px = px
            lbl = "1D outside"
            axes[ax_i].plot(
                col_px, row_px, FIT_MARKER, color="lime",
                markersize=9, markeredgewidth=1.5, markerfacecolor="none",
                label=lbl if lbl not in seen_lbls else "_nolegend_"
            )
            seen_lbls.add(lbl)
        if seen_lbls:
            axes[ax_i].legend(fontsize=7, loc="upper right",
                               framealpha=0.6, markerscale=0.8)

    dt_range_display = max(abs(dt_slice_qc.min()), dt_slice_qc.max())
    im_dt = axes[-1].imshow(dt_slice_qc, cmap="RdBu_r",
                             vmin=-dt_range_display, vmax=dt_range_display,
                             origin="upper")
    axes[-1].contour(dt_slice_qc, levels=[0], colors="k", linewidths=1.2)
    axes[-1].plot(cx, cy, "k+", markersize=10, markeredgewidth=1.5)

    for ch_i, ch in enumerate(receptor_channels):
        ch_peaks  = [r for r in outside_peaks if r["channel"] == ch]
        color     = CH_COLORS[ch_i % len(CH_COLORS)]
        seen_lbls = set()
        for pk in ch_peaks:
            px = peak_to_pixel(pk["theta_deg"], pk["centre_um"],
                               cx, cy, dt_slice_qc, H, W)
            if px is None:
                continue
            col_px, row_px = px
            lbl = f"Ch{ch} 1D outside"
            axes[-1].plot(
                col_px, row_px, FIT_MARKER, color=color,
                markersize=8, markeredgewidth=1.5, markerfacecolor="none",
                label=lbl if lbl not in seen_lbls else "_nolegend_"
            )
            seen_lbls.add(lbl)
    if outside_peaks:
        axes[-1].legend(fontsize=6, loc="upper right",
                         framealpha=0.6, markerscale=0.8)

    for th_deg, color, lbl in [(0, "yellow", "\u03b8=0\u00b0"),
                                 (90, "lime",   "\u03b8=90\u00b0")]:
        th = np.deg2rad(th_deg); dx, dy = np.cos(th), np.sin(th)
        margins = []
        if dx > 0:   margins.append((W - 1 - cx) / dx)
        elif dx < 0: margins.append(-cx / dx)
        if dy > 0:   margins.append((H - 1 - cy) / dy)
        elif dy < 0: margins.append(-cy / dy)
        ray_len = min(margins) if margins else max(H, W)
        axes[-1].annotate(
            "", xy=(cx + ray_len * dx, cy + ray_len * dy), xytext=(cx, cy),
            arrowprops=dict(arrowstyle="-|>", color=color, lw=2.0, mutation_scale=14)
        )
        lx = cx + ray_len * 0.65 * dx + (-dy) * 8
        ly = cy + ray_len * 0.65 * dy + ( dx) * 8
        axes[-1].text(lx, ly, lbl, color=color, fontsize=9,
                      fontweight="bold", ha="center", va="center")

    axes[-1].set_title(
        f"C{dt_channel}: Signed DT (\u00b5m)\n"
        f"\u03b8=0\u00b0 (yellow)  \u2022  \u03b8=90\u00b0 (green)  \u2022  "
        f"outside peaks \u25cb (per-channel colours)"
    )
    fig.colorbar(im_dt, ax=axes[-1], fraction=0.046, pad=0.04, label="\u00b5m")

    qc_path = os.path.join(output_folder, "QC_raw",
                            f"{sample_name}_QC_z{z_label_qc:04d}.png")
    fig.savefig(qc_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return qc_path


print(f"Generating QC plots for all {n_z} z-slices in parallel (N_JOBS={N_JOBS})...")
Parallel(n_jobs=N_JOBS, prefer="threads", verbose=0)(
    delayed(plot_qc_slice)(
        zi, stack, all_DT_stack, centroids, z_order,
        RECEPTOR_CHANNELS, CHANNEL_MARKERS, DT_CHANNEL,
        peak_fit_rows, H, W, OUTPUT_FOLDER, sample_name
    )
    for zi in range(n_z)
)
print(f"QC plots saved to: {FOLDER_QC}")

---
## 12. Outside Intensity Ratio

Uses the **membrane channel's 1D peak fits** (Cell 10) to define
per-θ-bin regions in the raw polar profile, then computes for every
channel what fraction of the total fitted-region intensity comes from
'outside' regions.

**Region definition** — for the global peak per θ-bin the global peak width is given by 
range_lo_um and range_hi_um.  The corresponding DT interval is
`[range_lo_um,  range_hi_um]` .
Only DT bins within that interval, at that exact θ-bin, contribute
intensity.

**Denominator** = sum of raw intensity over **all** fitted regions
(outside + boundary + inside).  θ-bins that were not fitted by the
membrane peaks are excluded from both numerator and denominator.


In [ ]:

def compute_outside_intensity_ratio(results_raw, peak_fit_rows,
                                     membrane_ch, receptor_channels,
                                     channel_markers, z_order,
                                     dt_bins_um):
    """
    For each z-slice and each channel, compute the fraction of raw polar-
    profile intensity that falls within 'outside' peak regions relative
    to all fitted peak regions (outside + boundary + inside).

    Region for a fitted θ-bin: DT bins in [range_lo_um, range_hi_um]
    from the peak detection result (FWHM-based global peak width).

    Parameters
    ----------
    results_raw    : dict {ch: list of (n_theta_z, n_bins) float32}  raw profiles
    peak_fit_rows  : list of dicts from Cell 10 (membrane + receptor fits)
    membrane_ch    : int  channel index used for region definitions
    receptor_channels : list of int
    channel_markers   : dict {ch: str}
    z_order           : list of z-slice indices
    dt_bins_um        : 1-D array of DT bin centres (µm)

    Returns
    -------
    ratio_rows : list of dicts — one row per (z_slice, channel)
    """
    all_channels = sorted(set(receptor_channels) | {membrane_ch})
    ratio_rows   = []

    for zi, z_idx in enumerate(z_order):

        # All membrane fits for this z-slice (every location)
        mem_fits = [
            r for r in peak_fit_rows
            if r["z_slice"] == z_idx and r["channel"] == membrane_ch
        ]

        n_outside   = sum(1 for f in mem_fits if f["location"] == "outside")
        n_boundary  = sum(1 for f in mem_fits if f["location"] == "boundary")
        n_inside    = sum(1 for f in mem_fits if f["location"] == "inside")
        n_fitted    = len(mem_fits)

        for ch in all_channels:
            prof = results_raw[ch][zi]   # (n_theta_z, n_bins) raw

            sum_outside  = 0.0
            sum_all      = 0.0

            for f in mem_fits:
                ti  = f["theta_idx"]

                # DT window from peak detection: [range_lo_um, range_hi_um]
                lo  = max(f["range_lo_um"], dt_bins_um[0])
                hi  = min(f["range_hi_um"], dt_bins_um[-1])

                bin_mask   = (dt_bins_um >= lo) & (dt_bins_um <= hi)
                region_sum = float(np.nansum(prof[ti, bin_mask]))

                sum_all += region_sum
                if f["location"] == "outside":
                    sum_outside += region_sum

            ratio = (sum_outside / sum_all
                     if sum_all > 0 and n_outside > 0
                     else np.nan)

            ratio_rows.append({
                "z_slice":          z_idx,
                "channel":          ch,
                "marker":           channel_markers[ch],
                "n_fitted":         n_fitted,
                "n_outside":        n_outside,
                "n_boundary":       n_boundary,
                "n_inside":         n_inside,
                "sum_intensity_outside": round(sum_outside, 4),
                "sum_intensity_all":     round(sum_all, 4),
                "outside_ratio":    round(ratio, 6) if np.isfinite(ratio) else np.nan,
                "outside_pct":      round(ratio * 100, 4) if np.isfinite(ratio) else np.nan,
            })

    return ratio_rows


def plot_outside_ratio(ratio_rows, z_order, channel_markers,
                        membrane_ch, receptor_channels,
                        output_folder, sample_name, channel_cmaps):
    """
    Two-panel figure saved to summary/:
      Top   : line plot of outside_pct vs z-slice for all channels
      Bottom: grouped bar plot of outside_pct per z-slice, one bar per channel
    """
    all_channels = sorted(set(receptor_channels) | {membrane_ch})
    LINE_COLORS  = ["tab:blue", "tab:orange", "tab:green",
                    "tab:red",  "tab:purple", "tab:brown",
                    "tab:pink", "tab:cyan"]

    z_vals_all = sorted(set(r["z_slice"] for r in ratio_rows))
    n_z_plot   = len(z_vals_all)
    n_ch_plot  = len(all_channels)
    bar_width  = 0.8 / n_ch_plot          # bars fill 80% of each z-slot
    bar_x_base = np.arange(n_z_plot)

    fig, (ax_line, ax_bar) = plt.subplots(
        2, 1, figsize=(max(10, n_z_plot * 0.35 + 3), 9),
        constrained_layout=True
    )
    fig.suptitle(
        f"{sample_name} \u2014 Outside intensity ratio\n"
        f"Regions from membrane Ch {membrane_ch} peak fits  "
        f"(FWHM-based peak range)",
        fontsize=11, fontweight="bold"
    )

    for ch_i, ch in enumerate(all_channels):
        marker   = channel_markers[ch]
        color    = LINE_COLORS[ch_i % len(LINE_COLORS)]
        ch_rows  = {r["z_slice"]: r["outside_pct"]
                    for r in ratio_rows if r["channel"] == ch}
        z_vals   = sorted(ch_rows.keys())
        pct_vals = [ch_rows[z] for z in z_vals]

        is_mem   = (ch == membrane_ch)
        lbl      = f"Ch {ch}: {marker}" + (" (membrane)" if is_mem else "")

        # ── Line plot ──
        ax_line.plot(z_vals, pct_vals,
                     color=color, ls="--" if is_mem else "-",
                     lw=2.0 if is_mem else 1.5,
                     marker="o", markersize=4, label=lbl)

        # ── Bar plot ──
        offset   = (ch_i - n_ch_plot / 2 + 0.5) * bar_width
        bar_vals = [ch_rows.get(z, np.nan) for z in z_vals_all]
        ax_bar.bar(
            bar_x_base + offset, bar_vals,
            width=bar_width, color=color,
            alpha=0.85, label=lbl,
            edgecolor="white", linewidth=0.3
        )

    # Line plot formatting
    ax_line.set_xlabel("Z-slice", fontsize=10)
    ax_line.set_ylabel("Outside intensity (%)", fontsize=10)
    ax_line.legend(fontsize=8, loc="upper right")
    ax_line.grid(True, alpha=0.3)
    ax_line.set_title("Line plot", fontsize=9, loc="left")

    # Bar plot formatting
    ax_bar.set_xticks(bar_x_base)
    ax_bar.set_xticklabels([str(z) for z in z_vals_all],
                            rotation=90, fontsize=7)
    ax_bar.set_xlabel("Z-slice", fontsize=10)
    ax_bar.set_ylabel("Outside intensity (%)", fontsize=10)
    ax_bar.legend(fontsize=8, loc="upper right")
    ax_bar.grid(True, axis="y", alpha=0.3)
    ax_bar.set_title("Bar plot (side-by-side per z-slice)", fontsize=9, loc="left")

    path = os.path.join(output_folder, "summary",
                         f"{sample_name}_outside_intensity_ratio.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Outside ratio plot saved: {path}")


def compute_receptor_vs_membrane(ratio_rows, membrane_ch, receptor_channels,
                                  channel_markers, z_order,
                                  output_folder, sample_name):
    """
    For each receptor channel and each z-slice, compute the normalised
    difference relative to the membrane channel:

        delta(z) = (outside_pct_receptor(z)- outside_pct_membrane(z))
                   / outside_pct_membrane(z)

    Then compute mean and SEM of delta across all z-slices for each
    receptor.  Slices where either value is NaN are excluded.

    Saves:
      summary/{sample}_receptor_vs_membrane.csv
      summary/{sample}_receptor_vs_membrane.png
    """
    LINE_COLORS = ["tab:blue", "tab:orange", "tab:green",
                   "tab:red",  "tab:purple", "tab:brown"]

    # Build lookup: {ch: {z_slice: outside_pct}}
    pct_by_ch = {}
    for r in ratio_rows:
        pct_by_ch.setdefault(r["channel"], {})[r["z_slice"]] = r["outside_pct"]

    mem_pct = pct_by_ch.get(membrane_ch, {})

    all_delta_rows = []
    summary_rows   = []

    for rec_ch in receptor_channels:
        rec_pct = pct_by_ch.get(rec_ch, {})
        marker  = channel_markers[rec_ch]
        deltas  = []

        for z in z_order:
            rp = rec_pct.get(z, np.nan)
            mp = mem_pct.get(z, np.nan)
            if np.isfinite(rp) and np.isfinite(mp) and mp != 0:
                delta = (rp- mp) / mp
            else:
                delta = np.nan
            deltas.append(delta)
            all_delta_rows.append({
                "z_slice":          z,
                "channel":          rec_ch,
                "marker":           marker,
                "outside_pct_receptor": round(rp, 4) if np.isfinite(rp) else np.nan,
                "outside_pct_membrane": round(mp, 4) if np.isfinite(mp) else np.nan,
                "delta_norm":       round(delta, 6) if np.isfinite(delta) else np.nan,
            })

        valid = [d for d in deltas if np.isfinite(d)]
        n_valid = len(valid)
        mean_d  = float(np.mean(valid))  if n_valid > 0 else np.nan
        sem_d   = (float(np.std(valid, ddof=1) / np.sqrt(n_valid))
                   if n_valid > 1 else np.nan)

        summary_rows.append({
            "channel":        rec_ch,
            "marker":         marker,
            "n_slices_valid": n_valid,
            "mean_delta":     round(mean_d, 6) if np.isfinite(mean_d) else np.nan,
            "sem_delta":      round(sem_d,  6) if np.isfinite(sem_d)  else np.nan,
            "mean_delta_pct": round(mean_d * 100, 4) if np.isfinite(mean_d) else np.nan,
            "sem_delta_pct":  round(sem_d  * 100, 4) if np.isfinite(sem_d)  else np.nan,
        })

    # ── Save CSVs ─────────────────────────────────────────────────────────
    delta_df   = pd.DataFrame(all_delta_rows)
    summary_df = pd.DataFrame(summary_rows)
    base = os.path.join(output_folder, "summary",
                         f"{sample_name}_receptor_vs_membrane")
    delta_df.to_csv(base + "_per_slice.csv", index=False)
    summary_df.to_csv(base + "_summary.csv", index=False)

    # ── Print summary ─────────────────────────────────────────────────────
    print(f"\n=== Receptor vs Membrane outside_pct: "
          f"\u0394 = (receptor \u2212 membrane) / membrane ===")
    print(f"{'ch':>3}  {'marker':>12}  {'n_valid':>7}  "
          f"{'mean_\u0394':>10}  {'SEM_\u0394':>10}  "
          f"{'mean_\u0394%':>10}  {'SEM_\u0394%':>10}")
    print("-" * 72)
    for r in summary_rows:
        print(f"{r['channel']:>3}  {r['marker']:>12}  {r['n_slices_valid']:>7}  "
              f"{r['mean_delta']:>10.4f}  {r['sem_delta']:>10.4f}  "
              f"{r['mean_delta_pct']:>10.2f}%  {r['sem_delta_pct']:>9.2f}%")

    # ── Figure: per-slice delta line + summary bar with error bars ────────
    fig, (ax_line, ax_bar) = plt.subplots(
        1, 2, figsize=(14, 5), constrained_layout=True
    )
    fig.suptitle(
        f"{sample_name} \u2014 Receptor vs Membrane outside_pct\n"
        f"\u0394 = (receptor \u2212 membrane) / membrane",
        fontsize=11, fontweight="bold"
    )

    # Left: per-slice delta for each receptor
    for ch_i, rec_ch in enumerate(receptor_channels):
        color  = LINE_COLORS[ch_i % len(LINE_COLORS)]
        marker = channel_markers[rec_ch]
        ch_del = [(r["z_slice"], r["delta_norm"])
                  for r in all_delta_rows if r["channel"] == rec_ch]
        ch_del.sort(key=lambda x: x[0])
        zz = [x[0] for x in ch_del]
        dd = [x[1] for x in ch_del]
        ax_line.plot(zz, dd, color=color, marker="o", markersize=4, lw=1.5,
                     label=f"Ch {rec_ch}: {marker}")

    ax_line.axhline(0, color="k", lw=0.8, ls="--", label="membrane baseline")
    ax_line.set_xlabel("Z-slice", fontsize=10)
    ax_line.set_ylabel("\u0394 = (rec \u2212 mem) / mem", fontsize=10)
    ax_line.legend(fontsize=8)
    ax_line.grid(True, alpha=0.3)
    ax_line.set_title("Per-slice \u0394", fontsize=9, loc="left")

    # Right: mean ± SEM bar for each receptor
    x_pos = np.arange(len(receptor_channels))
    for ch_i, rec_ch in enumerate(receptor_channels):
        sr    = next(r for r in summary_rows if r["channel"] == rec_ch)
        color = LINE_COLORS[ch_i % len(LINE_COLORS)]
        ax_bar.bar(
            x_pos[ch_i], sr["mean_delta"],
            yerr=sr["sem_delta"],
            color=color, alpha=0.85, width=0.5,
            capsize=6, edgecolor="k", linewidth=0.8,
            label=f"Ch {rec_ch}: {channel_markers[rec_ch]}  "
                  f"(n={sr['n_slices_valid']})"
        )

    ax_bar.axhline(0, color="k", lw=0.8, ls="--")
    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(
        [f"Ch {ch}\n{channel_markers[ch]}" for ch in receptor_channels],
        fontsize=9
    )
    ax_bar.set_ylabel("Mean \u0394 \u00b1 SEM", fontsize=10)
    ax_bar.legend(fontsize=8)
    ax_bar.grid(True, axis="y", alpha=0.3)
    ax_bar.set_title("Mean \u00b1 SEM across all z-slices", fontsize=9, loc="left")

    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\nCSVs saved:\n  {base}_per_slice.csv\n  {base}_summary.csv")
    print(f"Plot saved: {base}.png")

    return summary_df


# ── Run ───────────────────────────────────────────────────────────────────
ratio_rows = compute_outside_intensity_ratio(
    results,             # raw (un-normalised) profiles
    peak_fit_rows,
    membrane_ch=MEMBRANE_CH,
    receptor_channels=RECEPTOR_CHANNELS,
    channel_markers=CHANNEL_MARKERS,
    z_order=z_order,
    dt_bins_um=DT_BINS_UM,
)

# Save CSV
ratio_df   = pd.DataFrame(ratio_rows)
ratio_path = os.path.join(OUTPUT_FOLDER, "summary",
                           f"{sample_name}_outside_intensity_ratio.csv")
ratio_df.to_csv(ratio_path, index=False)
print(f"Outside ratio CSV saved: {ratio_path}")

# Print summary table
hdr = (f"{'z':>6}  {'ch':>3}  {'marker':>12}  "
       f"{'n_fit':>6}  {'n_out':>6}  {'n_bnd':>6}  {'n_in':>6}  "
       f"{'outside%':>10}")
print(f"\n{hdr}")
print("-" * len(hdr))
for r in ratio_rows:
    pct = (f"{r['outside_pct']:.2f}"
           if r['outside_pct'] is not None
           and not (isinstance(r['outside_pct'], float)
                    and np.isnan(r['outside_pct']))
           else "N/A")
    print(f"{r['z_slice']:>6}  {r['channel']:>3}  "
          f"{r['marker']:>12}  "
          f"{r['n_fitted']:>6}  {r['n_outside']:>6}  "
          f"{r['n_boundary']:>6}  {r['n_inside']:>6}  "
          f"{pct:>10}%")

# Line + bar plot
plot_outside_ratio(
    ratio_rows, z_order, CHANNEL_MARKERS,
    MEMBRANE_CH, RECEPTOR_CHANNELS,
    OUTPUT_FOLDER, sample_name, CHANNEL_CMAPS
)

# Receptor vs membrane difference analysis
receptor_vs_membrane_summary = compute_receptor_vs_membrane(
    ratio_rows,
    membrane_ch=MEMBRANE_CH,
    receptor_channels=RECEPTOR_CHANNELS,
    channel_markers=CHANNEL_MARKERS,
    z_order=z_order,
    output_folder=OUTPUT_FOLDER,
    sample_name=sample_name,
)


---
## 13. Region Masks & TIFF Stack Export

For every z-slice, a **location mask** is built directly from the membrane
channel's 1D peak fits (Cell 10) by mapping each fitted θ-bin to its
angular wedge of pixels in the 2D DT image.

**Mask construction (wedge method):**
1. Each pixel's angle from the centroid is computed and mapped to the
   nearest fitted θ-bin (same binning as the polar profile).
2. The pixel's 2D DT value must lie within `DT_RANGE_UM`; if not → 0.
3. The location class of that θ-bin's membrane peak fit is painted.

This gives gapless coverage: every pixel maps to exactly one θ-bin,
so no seams appear. Unfit θ-bins and out-of-range pixels stay at 0.

**Mask encoding (uint8):**
- `0` — outside `DT_RANGE_UM`, or θ-bin has no membrane fit
- `1` — inside region
- `2` — boundary region
- `3` — outside region

**Saved TIFF z-stacks** (to `results/tiff_export/`):
- Raw intensity stack per receptor + membrane channel (float32)
- Combined location mask stack (0/1/2/3, uint8)
- One binary mask stack per location class (outside / boundary / inside)

In [ ]:
# ── Output sub-folder ─────────────────────────────────────────────────────
FOLDER_TIFF = os.path.join(OUTPUT_FOLDER, "tiff_export")
os.makedirs(FOLDER_TIFF, exist_ok=True)


def build_location_mask(dt_slice, cx, cy, theta_angles, fits_for_slice,
                         dt_range_um, H, W):
    """
    Build a (H, W) uint8 location mask by directly mapping each fitted
    membrane θ-bin to its angular wedge of pixels, restricted to the
    DT_RANGE_UM band.  The mapping is gapless: every pixel is assigned
    to exactly one θ-bin (nearest bin-centre), so no seams appear between
    adjacent wedges.

    Pixel encoding
    --------------
    0  pixel is outside DT_RANGE_UM  OR  its θ-bin has no membrane fit
    1  inside   region
    2  boundary region
    3  outside  region

    Parameters
    ----------
    dt_slice        : (H, W) float32  — 2D signed DT in µm (from Cell 4)
    cx, cy          : float  — centroid pixel coordinates
    theta_angles    : 1-D array of bin-centre angles (degrees, 0→360, sorted)
    fits_for_slice  : list of fit dicts (membrane channel, all location classes)
                      Each dict must have keys: theta_idx, location
    dt_range_um     : (lo, hi)  — pixels outside this DT window → 0
    H, W            : image dimensions

    Returns
    -------
    mask : (H, W) uint8
    """
    LOC_VAL = {"outside": 3, "boundary": 2, "inside": 1}
    n_theta  = len(theta_angles)

    # Per-pixel angle from centroid → [0, 360)
    yy, xx    = np.mgrid[0:H, 0:W]
    angle_arr = np.degrees(
        np.arctan2((yy - cy).astype(np.float32),
                   (xx - cx).astype(np.float32))
    ) % 360.0

    # Map each pixel to nearest θ-bin index
    idx_map = np.searchsorted(theta_angles, angle_arr, side="right") - 1
    idx_map = np.clip(idx_map, 0, n_theta - 1)
    # Wrap-around correction: check if the next bin (wrapping at 360→0) is closer
    d_found = np.abs(angle_arr - theta_angles[idx_map])
    d_next  = np.abs(angle_arr - theta_angles[(idx_map + 1) % n_theta])
    wrap    = (idx_map == n_theta - 1)
    d_next[wrap] = np.abs(angle_arr[wrap] - (360.0 + theta_angles[0]))
    idx_map[d_next < d_found] = (idx_map[d_next < d_found] + 1) % n_theta

    # DT in-range mask
    dt_in_range = (dt_slice >= dt_range_um[0]) & (dt_slice <= dt_range_um[1])

    # Paint wedges: for each fitted θ-bin, colour all its pixels within DT range
    mask = np.zeros((H, W), dtype=np.uint8)
    for f in fits_for_slice:
        ti  = f["theta_idx"]
        val = LOC_VAL.get(f["location"], 0)
        mask[(idx_map == ti) & dt_in_range] = val

    return mask


# ── Build masks for all z-slices ──────────────────────────────────────────
print(f"Building location masks for {n_z} z-slices...")
print(f"Method: angular wedge per fitted θ-bin, restricted to DT_RANGE_UM={DT_RANGE_UM}")
print(f"Encoding: 0=no region  1=inside  2=boundary  3=outside")

location_masks = []   # list of (H, W) uint8, one per zi

for zi in range(n_z):
    z_idx        = z_order[zi]
    cy_zi, cx_zi = centroids[zi]

    # Membrane fits for this z-slice (all location classes)
    mem_fits_zi = [
        r for r in peak_fit_rows
        if r["z_slice"] == z_idx and r["channel"] == MEMBRANE_CH
    ]

    mask_zi = build_location_mask(
        all_DT_stack[zi], cx_zi, cy_zi,
        slice_theta_angles[zi],
        mem_fits_zi,
        DT_RANGE_UM, H, W
    )
    location_masks.append(mask_zi)

    if (zi + 1) % 10 == 0 or zi == n_z - 1:
        d = dict(zip(*np.unique(mask_zi, return_counts=True)))
        print(f"  z={z_idx:4d}  outside={d.get(3,0):6d} px  "
              f"boundary={d.get(2,0):6d} px  inside={d.get(1,0):6d} px  "
              f"no-region={d.get(0,0):6d} px  "
              f"fitted_theta={len(mem_fits_zi)}")

print(f"\nAll masks built.")

# ── Save TIFF z-stacks ────────────────────────────────────────────────────
print(f"\nSaving TIFF z-stacks to: {FOLDER_TIFF}")

channels_to_export = sorted(set(RECEPTOR_CHANNELS) | {MEMBRANE_CH})

# Raw intensity stacks — one per channel
for ch in channels_to_export:
    ci       = ch_order.index(ch)
    marker   = CHANNEL_MARKERS[ch]
    # stack[ci] has shape (n_z, H, W); convert to uint16 if values allow,
    # otherwise keep as float32
    raw_stack = stack[ci]   # (n_z, H, W) float32
    out_path  = os.path.join(
        FOLDER_TIFF,
        f"{sample_name}_C{ch:02d}_{marker}_raw.tif"
    )
    tifffile.imwrite(
        out_path,
        raw_stack.astype(np.float32),
        imagej=True,
        metadata={"axes": "ZYX"}
    )
    print(f"  Raw stack saved: {os.path.basename(out_path)}  "
          f"shape={raw_stack.shape}  dtype=float32")

# Combined location mask stack (0/1/2/3 encoding)
mask_stack = np.stack(location_masks, axis=0).astype(np.uint8)   # (n_z, H, W)
mask_path  = os.path.join(FOLDER_TIFF,
                            f"{sample_name}_location_mask.tif")
tifffile.imwrite(
    mask_path,
    mask_stack,
    imagej=True,
    metadata={"axes": "ZYX"}
)
print(f"  Location mask stack saved: {os.path.basename(mask_path)}  "
      f"shape={mask_stack.shape}  encoding: 0=none 1=inside 2=boundary 3=outside")

# Per-class binary masks
for loc_val, loc_name in [(3, "outside"), (2, "boundary"), (1, "inside")]:
    binary_stack = (mask_stack == loc_val).astype(np.uint8)   # (n_z, H, W)
    bin_path     = os.path.join(
        FOLDER_TIFF,
        f"{sample_name}_mask_{loc_name}.tif"
    )
    tifffile.imwrite(
        bin_path,
        binary_stack,
        imagej=True,
        metadata={"axes": "ZYX"}
    )
    n_px = int(binary_stack.sum())
    print(f"  Binary mask ({loc_name}) saved: {os.path.basename(bin_path)}  "
          f"non-zero px total={n_px}")

print(f"\nAll TIFF stacks saved to: {FOLDER_TIFF}")

---
## 14. Save Run Log

In [ ]:
# ── Finalise timer and write run log ──────────────────────────────────────
import datetime as _dt, time as _time, textwrap as _tw

_elapsed_s   = _time.perf_counter() - _RUN_START_TIME
_elapsed_str = str(_dt.timedelta(seconds=int(_elapsed_s)))
_end_stamp   = _dt.datetime.now()

# Update snapshot with runtime info known only at the end
_CONFIG_SNAPSHOT.update({
    "run_end_time":     _end_stamp.strftime("%H:%M:%S"),
    "total_runtime":    _elapsed_str,
    "total_runtime_s":  round(_elapsed_s, 1),
    "n_z_slices":       n_z,
    "z_range_used":     f"{z_order[0]} \u2013 {z_order[-1]}",
    "boundary_radius_um_mean":  round(float(slice_radii_um.mean()), 4),
    "boundary_radius_um_range": f"{slice_radii_um.min():.4f} \u2013 {slice_radii_um.max():.4f}",
    "theta_step_deg_range":     f"{min(360.0/n for n in slice_n_angles):.4f} \u2013 "
                                f"{max(360.0/n for n in slice_n_angles):.4f}",
    "sample_name":      sample_name,
})

# ── Format as a human-readable log file ───────────────────────────────────
_MAX_KEY = max(len(k) for k in _CONFIG_SNAPSHOT)
_header  = [
    "=" * 70,
    "  3D Cell Image Analysis — Run Log",
    "=" * 70,
]
_sections = {
    "Timing": [
        "run_date", "run_time", "run_end_time", "total_runtime", "total_runtime_s",
    ],
    "Paths": [
        "data_folder", "output_folder", "sample_name",
    ],
    "Dataset": [
        "n_z_slices", "z_range_used",
        "boundary_radius_um_mean", "boundary_radius_um_range",
        "theta_step_deg_range",
    ],
    "Voxel & Channels": [
        "voxel_xy_um", "voxel_z_um",
        "channel_markers", "dt_channel",
        "receptor_channels", "membrane_ch",
        "receptorA_ch", "receptorB_ch",
        "channel_range", "z_slice_range",
    ],
    "Preprocessing": [
        "window_size", "dt_smooth_sigma_px", "mask_channel", "mask_dilation_px",
    ],
    "Polar Profile": [
        "feature_size_um", "dt_range_um",
        "dt_bins_per_pixel", "n_dt_bins", "bin_width_um",
        "radial_step_px",
    ],
    "Peak Detection": [
        "peak_min_intensity", "peak_min_sep_um",
        "peak_rel_height", "boundary_tol_um",
    ],
    "Performance": [
        "n_jobs",
    ],
}

_lines = _header[:]
for section, keys in _sections.items():
    _lines.append("")
    _lines.append(f"  [{section}]")
    _lines.append("  " + "-" * 40)
    for k in keys:
        v = _CONFIG_SNAPSHOT.get(k, "N/A")
        _lines.append(f"  {k:<{_MAX_KEY}}  :  {v}")
_lines.append("")
_lines.append("=" * 70)

_log_text = "\n".join(_lines)

# ── Save to results folder ────────────────────────────────────────────────
_log_path = os.path.join(OUTPUT_FOLDER, "run_log.txt")
with open(_log_path, "w", encoding="utf-8") as _flog:
    _flog.write(_log_text + "\n")

print(_log_text)
print(f"\nLog saved to: {_log_path}")


---
## 15. Output Summary

All outputs are written to `{DATA_FOLDER}/results/` and its sub-folders:

**`summary/`** — CSVs and z-stack-wide summary PNGs

| File | Content |
|------|--------|
| `{sample}_{marker}_polar_profile.csv` | Long-format: z_slice, theta_deg, dt_um, mean_intensity, mean_intensity_norm |
| `{sample}_centroids.csv` | Cell centroid in px and µm per z-slice, boundary_radius_um, n_theta, theta_step_deg |
| `{sample}_zavg_polar_profile.png` | Z-averaged polar profile summary |
| `{sample}_zavg_diff_profile.png` | Z-averaged difference map summary |
| `{sample}_diff_profiles.csv` | Long-format: z_slice, theta_deg, dt_um, A−M, B−M, A−B |
| `{sample}_peak_fits.csv` | 1D peak fit results per z, channel, theta |
| `{sample}_outside_intensity_ratio.csv` | Per (z_slice, channel): outside/total intensity ratio |
| `{sample}_outside_intensity_ratio.png` | Line plot of outside% vs z-slice |

**`polar_profile/`** — per-slice heatmaps (raw + normalized)

| File | Content |
|------|--------|
| `{sample}_z{NNNN}_polar_profile.png` | Per-slice heatmap: raw (left) + normalized (right) |

**`diff_profile/`** — per-slice difference maps

| File | Content |
|------|--------|
| `{sample}_z{NNNN}_diff_profile.png` | Per-slice difference maps: A−M, B−M, A−B |

**`peaks/`** — per-slice 1D peak fit overlays

| File | Content |
|------|--------|
| `{sample}_z{NNNN}_peaks.png` | Per-slice peak fit (left) + region overlay (right) |

**`QC_raw/`** — per-slice raw image QC overlays

| File | Content |
|------|--------|
| `{sample}_QC_z{NNNN}.png` | Raw channel overlay with DT contour, θ reference rays, outside peak markers |

**`tiff_export/`** — TIFF z-stacks for downstream analysis

| File | Content |
|------|--------|
| `{sample}_C{NN}_{marker}_raw.tif` | Raw intensity z-stack per receptor + membrane channel (float32, ZYX) |
| `{sample}_location_mask.tif` | Location mask z-stack: 0=none, 1=inside, 2=boundary, 3=outside (uint8, ZYX) |
| `{sample}_mask_outside.tif` | Binary mask z-stack for outside region (uint8, ZYX) |
| `{sample}_mask_boundary.tif` | Binary mask z-stack for boundary region (uint8, ZYX) |
| `{sample}_mask_inside.tif` | Binary mask z-stack for inside region (uint8, ZYX) |